# Task 5: API Testing and Validation using Postman/Curl

**Objective:** Validate and test the Task 4 Flask Digit Classification API using industry-standard testing tools (Postman, curl) and an automated Python test runner.

**What this notebook does:**
1. Starts the Task 4 API in a background thread.
2. Runs a full suite of positive (functional) and negative (error-handling) tests against it, mirroring the accompanying Postman Collection (`postman_collection.json`) and curl script (`curl_commands.sh`).
3. Reports a clear PASS/FAIL result for every test, with a final summary.

**Result: 20/20 assertions passed (100.0%).**

The Postman Collection can be imported into Postman directly to run the same tests interactively with the GUI; this notebook (and `test_api_validation.py`) provide an equivalent automated/scriptable alternative.

## 1. The API Under Test (`app.py`, from Task 4)

In [1]:
"""
Task 4: Developing a Flask API for Deep Learning Models
==========================================================
Exposes the CNN digit-classification model trained in Task 1 (a from-scratch
NumPy CNN, saved as cnn_digits_model.pkl) as a REST API using Flask.

Endpoints:
  GET  /              - API information
  GET  /health        - health check
  POST /predict       - predict the digit for a single 8x8 grayscale image
  POST /predict/batch - predict digits for multiple images in one request

Run with:  python app.py
Then send requests to http://127.0.0.1:5000/
"""

import os
import pickle
import traceback
import numpy as np
from flask import Flask, request, jsonify

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # __file__ is not defined when this code runs inside a notebook cell
    # (Jupyter/Colab) rather than as a standalone script - fall back to the
    # current working directory in that case.
    BASE_DIR = os.getcwd()
MODEL_PATH = os.path.join(BASE_DIR, "cnn_digits_model.pkl")

app = Flask(__name__)

# ----------------------------------------------------------------------
# Model definition (must match the architecture used to train and save
# cnn_digits_model.pkl in Task 1) and inference-only forward pass.
# ----------------------------------------------------------------------


def im2col(x, kh, kw, stride=1, pad=0):
    N, H, W, C = x.shape
    if pad > 0:
        x = np.pad(x, ((0, 0), (pad, pad), (pad, pad), (0, 0)))
    out_h = (H + 2 * pad - kh) // stride + 1
    out_w = (W + 2 * pad - kw) // stride + 1
    cols = np.zeros((N, out_h, out_w, kh, kw, C), dtype=x.dtype)
    for i in range(kh):
        i_max = i + stride * out_h
        for j in range(kw):
            j_max = j + stride * out_w
            cols[:, :, :, i, j, :] = x[:, i:i_max:stride, j:j_max:stride, :]
    return cols.reshape(N, out_h, out_w, kh * kw * C), out_h, out_w


def conv_forward(x, W, b, stride=1, pad=1):
    N, H, Wd, C = x.shape
    k = W.shape[0]
    out_ch = W.shape[-1]
    cols, out_h, out_w = im2col(x, k, k, stride, pad)
    W_col = W.reshape(-1, out_ch)
    out = cols.reshape(N * out_h * out_w, -1) @ W_col + b
    return out.reshape(N, out_h, out_w, out_ch)


def relu(x):
    return np.maximum(0, x)


def maxpool_forward(x, size=2, stride=2):
    N, H, W, C = x.shape
    out_h, out_w = H // stride, W // stride
    x = x[:, :out_h * stride, :out_w * stride, :]
    out = np.zeros((N, out_h, out_w, C), dtype=x.dtype)
    for i in range(out_h):
        for j in range(out_w):
            window = x[:, i*stride:i*stride+size, j*stride:j*stride+size, :]
            out[:, i, j, :] = window.max(axis=(1, 2))
    return out


def softmax(logits):
    e = np.exp(logits - logits.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


class DigitCNN:
    """Loads the Task 1 CNN weights and runs inference only (no training)."""

    def __init__(self, weights_path):
        with open(weights_path, "rb") as f:
            self.w = pickle.load(f)

    def predict_proba(self, x):
        """x: numpy array of shape (N, 8, 8, 1), pixel values already in [0, 1]."""
        w = self.w
        x = conv_forward(x, w["conv1_W"], w["conv1_b"], stride=1, pad=1)
        x = relu(x)
        x = maxpool_forward(x, 2, 2)
        x = conv_forward(x, w["conv2_W"], w["conv2_b"], stride=1, pad=1)
        x = relu(x)
        x = maxpool_forward(x, 2, 2)
        x = x.reshape(x.shape[0], -1)
        x = x @ w["fc1_W"] + w["fc1_b"]
        x = relu(x)
        logits = x @ w["fc2_W"] + w["fc2_b"]
        return softmax(logits)


# Load the model once at startup, not on every request.
model = DigitCNN(MODEL_PATH)


# ----------------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------------

def parse_image(payload):
    """
    Validates and converts a single image payload into a (1, 8, 8, 1)
    float32 array normalized to [0, 1]. Raises ValueError with a clear
    message on any problem, which the route handlers turn into a 400.
    """
    if payload is None:
        raise ValueError("Missing 'image' field in request body.")

    arr = np.array(payload, dtype=np.float64)

    if arr.size != 64:
        raise ValueError(
            f"Expected an 8x8 (64-value) grayscale image, got {arr.size} values."
        )

    arr = arr.reshape(8, 8)

    if np.isnan(arr).any():
        raise ValueError("Image contains non-numeric or missing values.")

    # Accept either raw 0-16 pixel scale (like the original digits dataset)
    # or already-normalized 0-1 scale, and normalize consistently to [0, 1].
    if arr.max() > 1.0:
        if arr.max() > 16.0 or arr.min() < 0.0:
            raise ValueError("Pixel values must be within [0, 16] (or already normalized to [0, 1]).")
        arr = arr / 16.0
    elif arr.min() < 0.0:
        raise ValueError("Pixel values must not be negative.")

    return arr.reshape(1, 8, 8, 1).astype(np.float32)


def format_prediction(probs_row):
    probs_row = probs_row.tolist()
    pred_digit = int(np.argmax(probs_row))
    return {
        "predicted_digit": pred_digit,
        "confidence": round(probs_row[pred_digit], 4),
        "probabilities": {str(i): round(p, 4) for i, p in enumerate(probs_row)},
    }


# ----------------------------------------------------------------------
# Routes
# ----------------------------------------------------------------------

@app.route("/", methods=["GET"])
def index():
    return jsonify({
        "name": "Digit Classification API",
        "description": "REST API serving the Task 1 from-scratch CNN digit classifier.",
        "endpoints": {
            "GET /health": "Health check.",
            "POST /predict": "Predict the digit for one 8x8 grayscale image. "
                              "Body: {\"image\": [64 numbers, 0-16 or 0-1, row-major 8x8]}",
            "POST /predict/batch": "Predict digits for multiple images. "
                                    "Body: {\"images\": [[64 numbers], [64 numbers], ...]}",
        },
    })


@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "model_loaded": model is not None})


@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json(silent=True)
        if data is None:
            return jsonify({"error": "Request body must be valid JSON."}), 400

        image = parse_image(data.get("image"))
        probs = model.predict_proba(image)[0]
        return jsonify(format_prediction(probs)), 200

    except ValueError as e:
        return jsonify({"error": str(e)}), 400
    except Exception as e:
        app.logger.error("Unexpected error in /predict: %s\n%s", e, traceback.format_exc())
        return jsonify({"error": "Internal server error while generating prediction."}), 500


@app.route("/predict/batch", methods=["POST"])
def predict_batch():
    try:
        data = request.get_json(silent=True)
        if data is None:
            return jsonify({"error": "Request body must be valid JSON."}), 400

        images_payload = data.get("images")
        if not isinstance(images_payload, list) or len(images_payload) == 0:
            return jsonify({"error": "'images' must be a non-empty list of 64-value images."}), 400
        if len(images_payload) > 100:
            return jsonify({"error": "Batch size limited to 100 images per request."}), 400

        results = []
        for idx, img_payload in enumerate(images_payload):
            try:
                image = parse_image(img_payload)
                probs = model.predict_proba(image)[0]
                results.append(format_prediction(probs))
            except ValueError as e:
                results.append({"error": f"image[{idx}]: {e}"})

        return jsonify({"count": len(results), "results": results}), 200

    except Exception as e:
        app.logger.error("Unexpected error in /predict/batch: %s\n%s", e, traceback.format_exc())
        return jsonify({"error": "Internal server error while generating predictions."}), 500


# ----------------------------------------------------------------------
# Error handlers for common HTTP errors
# ----------------------------------------------------------------------

@app.errorhandler(404)
def not_found(e):
    return jsonify({"error": "Endpoint not found. See GET / for a list of available endpoints."}), 404


@app.errorhandler(405)
def method_not_allowed(e):
    return jsonify({"error": "Method not allowed for this endpoint."}), 405


@app.errorhandler(500)
def server_error(e):
    return jsonify({"error": "Internal server error."}), 500

# Note: the standalone app.py file ends with:
#     if __name__ == "__main__":
#         app.run(host="0.0.0.0", port=5000, debug=False)
# That is intentionally left out here - __name__ is also "__main__"
# inside a notebook cell, so including it would call the BLOCKING Flask
# dev server immediately and freeze this cell forever. The next cell
# starts the server properly, in a background thread, instead.

## 2. Start the API (background thread, for this notebook demo)

In [1]:
import threading, time

def run_server():
    app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(1.5)
print("API is running on http://127.0.0.1:5000")

API is running on http://127.0.0.1:5000


## 3. Automated Test Suite

The full test script (also provided standalone as `test_api_validation.py`), covering the same positive and negative test cases as `postman_collection.json`. Shown here for reference only - it is run for real in the next section.

```python
"""
Task 5: API Testing and Validation using Postman/Curl
=========================================================
Automated test suite for the Task 4 Flask Digit Classification API,
mirroring every request in postman_collection.json (positive/functional
tests and negative/error-handling tests). Each test sends a real HTTP
request to the running API, validates the status code and JSON response
structure, and prints a clear PASS/FAIL line - producing the same kind of
pass/fail report a Postman Collection Runner (or `newman run`) would.

Run the API first (`python app.py`), then run this script:
    python test_api_validation.py
"""

import json
import sys
import requests
from sklearn.datasets import load_digits

BASE_URL = "http://127.0.0.1:5000"

results = []


def record(name, passed, detail=""):
    results.append({"name": name, "passed": passed, "detail": detail})
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {name}" + (f"  -  {detail}" if detail else ""))


def check(name, condition, detail=""):
    record(name, bool(condition), detail)


SAMPLE_ZERO = [
    0, 0, 5, 13, 9, 1, 0, 0,
    0, 0, 13, 15, 10, 15, 5, 0,
    0, 3, 15, 2, 0, 11, 8, 0,
    0, 4, 12, 0, 0, 8, 8, 0,
    0, 5, 8, 0, 0, 9, 8, 0,
    0, 4, 11, 0, 1, 12, 7, 0,
    0, 2, 14, 5, 10, 12, 0, 0,
    0, 0, 6, 13, 10, 0, 0, 0,
]


def get_real_digit(digit):
    d = load_digits()
    idx = list(d.target).index(digit)
    return d.images[idx].flatten().tolist()


print("=" * 70)
print("SECTION 1: Positive (Functional) Tests")
print("=" * 70)

# --- GET / ---
r = requests.get(f"{BASE_URL}/")
check("GET / returns 200", r.status_code == 200, f"status={r.status_code}")
body = r.json()
check("GET / response has 'name' and 'endpoints'",
      "name" in body and "endpoints" in body, json.dumps(body)[:80] + "...")

# --- GET /health ---
r = requests.get(f"{BASE_URL}/health")
check("GET /health returns 200", r.status_code == 200, f"status={r.status_code}")
check("GET /health reports model_loaded=true",
      r.json().get("model_loaded") is True, json.dumps(r.json()))

# --- POST /predict (valid) ---
r = requests.post(f"{BASE_URL}/predict", json={"image": SAMPLE_ZERO})
check("POST /predict (valid image) returns 200", r.status_code == 200, f"status={r.status_code}")
body = r.json()
check("POST /predict response has predicted_digit/confidence/probabilities",
      all(k in body for k in ("predicted_digit", "confidence", "probabilities")))
check("POST /predict correctly predicts digit 0",
      body.get("predicted_digit") == 0, f"predicted={body.get('predicted_digit')}, confidence={body.get('confidence')}")

# --- POST /predict/batch (valid, real digits 3, 7, 8) ---
images = [get_real_digit(3), get_real_digit(7), get_real_digit(8)]
r = requests.post(f"{BASE_URL}/predict/batch", json={"images": images})
check("POST /predict/batch (valid) returns 200", r.status_code == 200, f"status={r.status_code}")
body = r.json()
check("POST /predict/batch returns count == 3", body.get("count") == 3, json.dumps(body.get("count")))
preds = [item["predicted_digit"] for item in body.get("results", [])]
check("POST /predict/batch correctly predicts [3, 7, 8]", preds == [3, 7, 8], f"got {preds}")

print()
print("=" * 70)
print("SECTION 2: Negative Tests (Error Handling)")
print("=" * 70)

# --- Missing 'image' field ---
r = requests.post(f"{BASE_URL}/predict", json={})
check("POST /predict (missing image) returns 400", r.status_code == 400, f"status={r.status_code}")
check("POST /predict (missing image) response has 'error'", "error" in r.json(), json.dumps(r.json()))

# --- Wrong-size image array ---
r = requests.post(f"{BASE_URL}/predict", json={"image": [1, 2, 3]})
check("POST /predict (wrong size) returns 400", r.status_code == 400, f"status={r.status_code}")

# --- Invalid JSON body ---
r = requests.post(f"{BASE_URL}/predict", data="not-json", headers={"Content-Type": "application/json"})
check("POST /predict (invalid JSON) returns 400", r.status_code == 400, f"status={r.status_code}")

# --- Out-of-range pixel values ---
bad_pixels = [999] * 64
r = requests.post(f"{BASE_URL}/predict", json={"image": bad_pixels})
check("POST /predict (out-of-range pixels) returns 400", r.status_code == 400, f"status={r.status_code}")

# --- Empty batch list ---
r = requests.post(f"{BASE_URL}/predict/batch", json={"images": []})
check("POST /predict/batch (empty list) returns 400", r.status_code == 400, f"status={r.status_code}")

# --- Batch with one bad image among good ones (per-item isolation) ---
r = requests.post(f"{BASE_URL}/predict/batch", json={"images": [SAMPLE_ZERO, [1, 2, 3]]})
body = r.json()
check("POST /predict/batch (mixed valid/invalid) still returns 200", r.status_code == 200, f"status={r.status_code}")
check("POST /predict/batch isolates the bad image's error",
      "error" in body["results"][1] and "predicted_digit" in body["results"][0],
      json.dumps(body))

# --- Unknown endpoint ---
r = requests.get(f"{BASE_URL}/nonexistent")
check("GET /nonexistent returns 404", r.status_code == 404, f"status={r.status_code}")

# --- Wrong HTTP method ---
r = requests.get(f"{BASE_URL}/predict")
check("GET /predict (wrong method) returns 405", r.status_code == 405, f"status={r.status_code}")

# ------------------------------------------------------------------
print()
print("=" * 70)
print("TEST SUMMARY")
print("=" * 70)
total = len(results)
passed = sum(1 for r in results if r["passed"])
failed = total - passed
print(f"Total tests : {total}")
print(f"Passed      : {passed}")
print(f"Failed      : {failed}")
print(f"Pass rate   : {passed/total*100:.1f}%")

with open("test_results.json", "w") as f:
    json.dump({"total": total, "passed": passed, "failed": failed, "results": results}, f, indent=2)

if failed > 0:
    sys.exit(1)

```

## 4. Running the Tests

In [1]:
exec(open("test_api_validation.py").read())

SECTION 1: Positive (Functional) Tests
[PASS] GET / returns 200  -  status=200
[PASS] GET / response has 'name' and 'endpoints'  -  {"description": "REST API serving the Task 1 from-scratch CNN digit classifier."...
[PASS] GET /health returns 200  -  status=200
[PASS] GET /health reports model_loaded=true  -  {"model_loaded": true, "status": "ok"}
[PASS] POST /predict (valid image) returns 200  -  status=200
[PASS] POST /predict response has predicted_digit/confidence/probabilities
[PASS] POST /predict correctly predicts digit 0  -  predicted=0, confidence=0.9998
[PASS] POST /predict/batch (valid) returns 200  -  status=200
[PASS] POST /predict/batch returns count == 3  -  3
[PASS] POST /predict/batch correctly predicts [3, 7, 8]  -  got [3, 7, 8]

SECTION 2: Negative Tests (Error Handling)
[PASS] POST /predict (missing image) returns 400  -  status=400
[PASS] POST /predict (missing image) response has 'error'  -  {"error": "Missing 'image' field in request body."}
[PASS] POST /predict

## 5. Results Summary

| Metric | Value |
|---|---|
| Total assertions | 20 |
| Passed | 20 |
| Failed | 0 |
| Pass rate | 100.0% |

**Observations:**
- Every endpoint returns the correct HTTP status code for both valid and invalid input, and every response body is valid, well-structured JSON in every case tested - including all negative/error cases.
- The batch endpoint correctly isolates a single malformed image's error without failing the other, valid images in the same request (verified directly by the mixed valid/invalid batch test).
- No test produced a 500 Internal Server Error, meaning all anticipated failure modes are caught by explicit input validation rather than falling through to the generic exception handler.
- The equivalent Postman Collection (`postman_collection.json`) can be imported into Postman to run and inspect the exact same requests interactively, with the same built-in test assertions shown in Postman's own test-results panel.